# Annexe B — Cahier de code, Chapitre 2
## Les moments d'image

Ce notebook accompagne le chapitre 2 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Masque synthétique : une ellipse pleine (forme à décrire)
import numpy as np
from skimage import measure, draw
import matplotlib.pyplot as plt

mask = np.zeros((220, 260), dtype=np.uint8)
rr, cc = draw.ellipse(110, 130, 55, 95, rotation=np.deg2rad(20))
mask[rr, cc] = 1

prop = measure.regionprops(mask)[0]      # une seule région
plt.imshow(mask, cmap="gray"); plt.title("masque d'entrée"); plt.show()

## 2.1 — Moments bruts

In [ ]:
# M[i,j] = Σ x^j · y^i · I(x,y)
M = measure.moments(mask, order=3)
print("aire M00 =", M[0, 0])

## 2.2 — Centroïde

In [ ]:
# centre de masse : (M10/M00, M01/M00)
cy, cx = M[1, 0] / M[0, 0], M[0, 1] / M[0, 0]
print(cy, cx)

## 2.3 — Moments centraux

In [ ]:
# moments calculés depuis le centroïde (invariants à la translation)
mu = measure.moments_central(mask, order=3)
print(mu[2, 0], mu[0, 2], mu[1, 1])

## 2.4 — Moments normalisés

In [ ]:
# moments centraux divisés par une puissance de l'aire (invariants à l'échelle)
nu = measure.moments_normalized(mu, order=3)
print(nu[2, 0], nu[0, 2])

## 2.5 — Orientation principale

In [ ]:
# angle de l'axe de moindre inertie
theta = 0.5 * np.arctan2(2 * mu[1, 1], mu[2, 0] - mu[0, 2])
print(np.rad2deg(theta), "ou directement :", np.rad2deg(prop.orientation))

## 2.6 — Ellipse équivalente

In [ ]:
# l'ellipse de mêmes moments d'ordre 2
print("grand axe :", prop.axis_major_length)
print("petit axe :", prop.axis_minor_length)
print("angle     :", np.rad2deg(prop.orientation))

## 2.7 — Les sept invariants de Hu

In [ ]:
# 7 combinaisons invariantes à translation, échelle, rotation
hu = measure.moments_hu(nu)
print(hu)

## 2.8 — Moments pondérés par l'intensité

In [ ]:
# mêmes moments mais sur les niveaux de gris (pas un masque binaire)
from skimage import data
g = data.coins()                 # image déjà en niveaux de gris (uint8)
Mi = measure.moments(g)
print("centroïde pondéré :", Mi[1, 0] / Mi[0, 0], Mi[0, 1] / Mi[0, 0])